In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
import sys
sys.version

'3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]'

In [4]:
spark = (
    SparkSession.builder
    .appName("coingecko")
    .master("spark://spark-master:7077")   # 🔥 WAJIB
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.0


In [5]:
spark.sparkContext.master

'spark://spark-master:7077'

In [6]:
schema = StructType([
StructField("crypto_id", StringType()),
StructField("price_usd", DoubleType()),
StructField("market_cap", DoubleType()),
StructField("volume_24h", DoubleType()),
StructField("return_24h", DoubleType()),
StructField("volatility_proxy", DoubleType()),
StructField("event_time", StringType())
])

In [7]:
df = spark.readStream \
.format("kafka") \
.option("kafka.bootstrap.servers", "kafka:29092") \
.option("subscribe", "coingecko") \
.load()


parsed = df.select(
from_json(col("value").cast("string"), schema).alias("data")
).select("data.*") \
.withColumn("event_time", to_timestamp("event_time"))

In [8]:
# === DATA QUALITY FILTER ===
clean = parsed.filter(col("price_usd") > 0)

In [10]:
# === WINDOW AGGREGATION ===
agg = clean \
.withWatermark("event_time", "30 minutes") \
.groupBy(
window(col("event_time"), "5 minutes"),
col("crypto_id")
) \
.agg(
avg("price_usd").alias("avg_price"),
max("price_usd").alias("max_price"),
avg("return_24h").alias("avg_return"),
avg("volatility_proxy").alias("avg_volatility")
)

In [11]:
agg

DataFrame[window: struct<start:timestamp,end:timestamp>, crypto_id: string, avg_price: double, max_price: double, avg_return: double, avg_volatility: double]

In [12]:
query = agg.writeStream \
.format("parquet") \
.option("path", "s3a://data-lake/curated/coingecko") \
.option("checkpointLocation", "/tmp/crypto_cp") \
.outputMode("append") \
.start()